# WhyGame MCP Slice

This notebook is the Phase 14 proof artifact for the thin FastMCP surface and the successor-local WhyGame relationship adapter.

It demonstrates:

1. real WhyGame relationship import through the MCP-callable boundary;
2. candidate review over the imported records;
3. explicit graph promotion;
4. governed-bundle export with visible imported provenance.

Related assets:

- `src/onto_canon6/mcp_server.py`
- `src/onto_canon6/adapters/whygame_service.py`
- `tests/integration/test_mcp_server.py`
- `tests/adapters/test_whygame_service.py`
- `docs/adr/0015-recover-phase-14-through-a-thin-mcp-surface-and-a-whygame-relationship-adapter.md`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import json
from tempfile import TemporaryDirectory

from onto_canon6 import mcp_server

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
fixture_path = repo_root / 'tests' / 'fixtures' / 'whygame_relationship_facts.json'
facts = json.loads(fixture_path.read_text(encoding='utf-8'))
sorted(mcp_server.mcp._tool_manager._tools.keys())


['canon6_apply_overlay',
 'canon6_export_governed_bundle',
 'canon6_import_whygame_relationships',
 'canon6_list_candidates',
 'canon6_list_proposals',
 'canon6_promote_candidate',
 'canon6_review_candidate',
 'canon6_review_proposal']

In [2]:
temp_dir = TemporaryDirectory()
review_db_path = Path(temp_dir.name) / 'review.sqlite3'

import_result = mcp_server.canon6_import_whygame_relationships(
    facts=facts,
    submitted_by='analyst:notebook',
    source_ref='whygame://notebook/ai-military',
    source_label='WhyGame notebook fixture',
    artifact_uri='artifact://whygame/notebook/ai-military',
    review_db_path=str(review_db_path),
)
import_result


{'profile': {'profile_id': 'whygame_minimal_strict',
  'profile_version': '0.1.0'},
 'artifact': {'artifact_id': 'art_18fe852e3a75451aaf15efdf',
  'artifact_kind': 'analysis_result',
  'uri': 'artifact://whygame/notebook/ai-military',
  'label': 'WhyGame notebook fixture',
  'metadata': {'adapter': 'whygame_relationship_import',
   'fact_count': 2,
   'profile_id': 'whygame_minimal_strict',
   'profile_version': '0.1.0',
   'source_ref': 'whygame://notebook/ai-military'},
  'fingerprint': None,
  'created_at': '2026-03-18T20:43:29.276083+00:00'},
 'submissions': [{'candidate': {'candidate_id': 'cand_28401efbab724d01bfd117ea',
    'profile': {'profile_id': 'whygame_minimal_strict',
     'profile_version': '0.1.0'},
    'validation_status': 'valid',
    'review_status': 'pending_review',
    'payload_hash': 'sha256:ff0a6332e3c3d36c6210eb69e4204f95f81bd3189dfd76ba006f3e376e2bfdf5',
    'payload': {'predicate': 'whygame:relationship',
     'roles': {'relationship_label': [{'kind': 'value',

In [3]:
candidate_id = import_result['submissions'][0]['candidate']['candidate_id']

listed_candidates = mcp_server.canon6_list_candidates(
    review_db_path=str(review_db_path),
)
reviewed_candidate = mcp_server.canon6_review_candidate(
    candidate_id=candidate_id,
    decision='accepted',
    actor_id='analyst:notebook-reviewer',
    review_db_path=str(review_db_path),
)
listed_candidates, reviewed_candidate


([{'candidate_id': 'cand_28401efbab724d01bfd117ea',
   'profile': {'profile_id': 'whygame_minimal_strict',
    'profile_version': '0.1.0'},
   'validation_status': 'valid',
   'review_status': 'pending_review',
   'payload_hash': 'sha256:ff0a6332e3c3d36c6210eb69e4204f95f81bd3189dfd76ba006f3e376e2bfdf5',
   'payload': {'predicate': 'whygame:relationship',
    'roles': {'relationship_label': [{'kind': 'value',
       'value': 'supports',
       'value_kind': 'string'}],
     'source_concept': [{'entity_id': 'ent:whygame:ai_integration',
       'entity_type': 'whygame:Concept',
       'kind': 'entity',
       'name': 'AI integration'}],
     'target_concept': [{'entity_id': 'ent:whygame:military_modernization',
       'entity_type': 'whygame:Concept',
       'kind': 'entity',
       'name': 'military modernization'}]}},
   'normalized_payload': {'predicate': 'whygame:relationship',
    'roles': {'relationship_label': [{'kind': 'value',
       'value': 'supports',
       'value_kind': 'str

In [4]:
promoted = mcp_server.canon6_promote_candidate(
    candidate_id=candidate_id,
    actor_id='analyst:notebook-reviewer',
    review_db_path=str(review_db_path),
)
bundle = mcp_server.canon6_export_governed_bundle(
    candidate_ids=[candidate_id],
    review_db_path=str(review_db_path),
)
{'promoted': promoted, 'bundle': bundle}


{'promoted': {'assertion': {'assertion_id': 'gassert_d44e81f43de8a717e5dfd9d6',
   'source_candidate_id': 'cand_28401efbab724d01bfd117ea',
   'profile': {'profile_id': 'whygame_minimal_strict',
    'profile_version': '0.1.0'},
   'predicate': 'whygame:relationship',
   'normalized_body': {'predicate': 'whygame:relationship',
    'roles': {'relationship_label': [{'kind': 'value',
       'value': 'supports',
       'value_kind': 'string'}],
     'source_concept': [{'entity_id': 'ent:whygame:ai_integration',
       'entity_type': 'whygame:Concept',
       'kind': 'entity',
       'name': 'AI integration'}],
     'target_concept': [{'entity_id': 'ent:whygame:military_modernization',
       'entity_type': 'whygame:Concept',
       'kind': 'entity',
       'name': 'military modernization'}]}},
   'claim_text': 'AI integration supports military modernization.',
   'promoted_by': 'analyst:notebook-reviewer',
   'promoted_at': '2026-03-18T20:43:29.341475+00:00'},
  'role_fillers': [{'assertion_

In [5]:
assert import_result['artifact']['artifact_kind'] == 'analysis_result'
assert reviewed_candidate['review_status'] == 'accepted'
assert promoted['assertion']['source_candidate_id'] == candidate_id
assert bundle['summary']['total_candidates'] == 1
assert bundle['candidate_bundles'][0]['artifacts'][0]['uri'] == 'artifact://whygame/notebook/ai-military'
'phase14 proof complete'


'phase14 proof complete'